# Stage 1 — 11: OOF-guided A7 + forensic fusion

This notebook does **no training**.

Fusion selection is separated into two domains:

- **OOF proxy domain:** 09B grouped V-JEPA probe OOF + 10 forensic OOF. Every
  row is predicted by a model that did not train on that row's group.
- **retention domain:** original A7 predictions on the untouched fixed DLC val
  + forensic fold-ensemble predictions.

The notebook learns a recipe on OOF, transfers the same recipe to A7, and
accepts it only if the fixed validation split retains A7's hard labels at
threshold 0.5.

Two fusion families are searched:

1. simple probability blend,
2. uncertainty-gated blend: forensic evidence is allowed to influence V-JEPA
   only when the base probability is close to 0.5.

This is deliberately more conservative than experiment 06, while still giving
the auxiliary branch enough room to change genuinely uncertain cases.


## 1. Setup


In [ ]:
from __future__ import annotations

import copy
import gc
import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path

REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage1-sangchun"

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)
except ModuleNotFoundError:
    print("Not running in Google Colab; Drive mount skipped.")

if IN_COLAB:
    REPO_ROOT = Path("/content/Blackbox-Detection")
    if not (REPO_ROOT / ".git").is_dir():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", BRANCH,
            "--single-branch", REPO_URL, str(REPO_ROOT),
        ], check=True)
    else:
        current_branch = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "branch", "--show-current"],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if current_branch != BRANCH:
            subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", BRANCH], check=True)
        dirty = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "status", "--porcelain"],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if dirty:
            print("WARNING: local repo has changes; git pull skipped.")
        else:
            subprocess.run(
                ["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH],
                check=True,
            )
else:
    REPO_ROOT = Path.cwd().resolve()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "pyproject.toml").is_file():
        raise FileNotFoundError("Run this notebook inside Blackbox-Detection repository.")

os.chdir(REPO_ROOT)

# Keep Colab's binary scientific stack intact. Install only the Stage 1 extras
# and then this repository editable with --no-deps, matching the current notebooks.
COLAB_EXTRAS = [
    "av>=15,<17", "timm==1.0.15", "fvcore==0.1.5.post20221221",
    "iopath==0.1.10", "yacs==0.1.8", "einops==0.8.1",
    "omegaconf==2.3.0", "hydra-core==1.3.2", "easydict==1.13",
]
if IN_COLAB:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed", *COLAB_EXTRAS,
    ], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_ROOT)
], check=True)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import yaml

from blackbox_detection.utils import load_checkpoint, seed_everything

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATASET_ROOT = DRIVE_PROJECT_ROOT / "DATASET"
DLC_ROOT = DATASET_ROOT / "DLC-2021"
DLC_SPLIT_CSV = DLC_ROOT / "dlc_split.csv"
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "outputs" / "stage1"
CONFIG_DIR = REPO_ROOT / "configs" / "stage1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    required = {"DLC_ROOT": DLC_ROOT, "DLC_SPLIT_CSV": DLC_SPLIT_CSV}
    missing = [f"{k}: {v}" for k, v in required.items() if not v.exists()]
    if missing:
        raise FileNotFoundError("Missing required Drive paths:\n  " + "\n  ".join(missing))

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print("repo   :", REPO_ROOT)
print("branch :", BRANCH)
print("commit :", GIT_COMMIT)
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu    :", torch.cuda.get_device_name(0))


In [ ]:
from blackbox_detection.stage1.evaluator import (
    evaluate_predictions, probabilities_to_labels, save_predictions,
)
from blackbox_detection.utils.metrics import stage1_score

RUN_DIR = OUTPUT_ROOT / "11_a7_forensic_fusion"
RUN_DIR.mkdir(parents=True, exist_ok=True)
A7_VAL_PATH = OUTPUT_ROOT / "09a_a7_hard_validation" / "a7_center_val_predictions.csv"
A7_FALLBACK = OUTPUT_ROOT / "dlc" / "vjepa2_1_b" / "val_predictions.csv"
VJEPA_OOF_PATH = OUTPUT_ROOT / "09b_grouped_probe_cv" / "oof_predictions.csv"
FOLD_ASSIGNMENTS_PATH = OUTPUT_ROOT / "09b_grouped_probe_cv" / "fold_assignments.csv"
FORENSIC_ROOT = OUTPUT_ROOT / "10_forensic_groupcv"
FORENSIC_MODELS = ["bayar_resnet18", "frequency", "lcdf"]

if not A7_VAL_PATH.is_file():
    if A7_FALLBACK.is_file():
        print("09A A7 predictions not found; using original A7 val_predictions.csv")
        A7_VAL_PATH = A7_FALLBACK
    else:
        raise FileNotFoundError("No A7 validation prediction file found.")
missing = [p for p in [VJEPA_OOF_PATH, FOLD_ASSIGNMENTS_PATH] if not p.is_file()]
if missing:
    raise FileNotFoundError(f"Run 09B first. Missing: {missing}")
print("A7 val :", A7_VAL_PATH)
print("VJ OOF :", VJEPA_OOF_PATH)


## 2. Load and align OOF predictions


In [ ]:
def load_pred(path: Path, name: str) -> pd.DataFrame:
    frame = pd.read_csv(path)
    required = {"video_id", "label", "dataset", "prob_rerecorded"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{path} missing columns: {sorted(missing)}")
    out = frame[["video_id", "label", "dataset", "prob_rerecorded"]].copy()
    return out.rename(columns={"prob_rerecorded": f"p_{name}"})

base_oof = load_pred(VJEPA_OOF_PATH, "base")
folds = pd.read_csv(FOLD_ASSIGNMENTS_PATH)[["video_id", "fold"]]
oof_tables = {}
for model_name in FORENSIC_MODELS:
    path = FORENSIC_ROOT / model_name / "oof_predictions.csv"
    if not path.is_file():
        print("skip missing forensic OOF:", path)
        continue
    aux = load_pred(path, model_name)
    merged = base_oof.merge(
        aux, on=["video_id", "label", "dataset"], how="inner", validate="one_to_one"
    ).merge(folds, on="video_id", how="left", validate="one_to_one")
    if len(merged) != len(base_oof):
        raise RuntimeError(f"{model_name}: OOF coverage mismatch {len(merged)} != {len(base_oof)}")
    if merged["fold"].isna().any():
        raise RuntimeError(f"{model_name}: missing fold ids")
    oof_tables[model_name] = merged.reset_index(drop=True)
if not oof_tables:
    raise FileNotFoundError("No forensic OOF predictions found. Run notebook 10 first.")
print("available forensic models:", list(oof_tables))


## 3. Search fixed-threshold fusion recipes on grouped OOF


In [ ]:
ALPHAS = np.round(np.arange(0.05, 0.81, 0.05), 2)
MARGINS = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
THRESHOLD = 0.5
MAX_WORST_FOLD_DROP = 0.01

def macro_f1(labels, probs, threshold=0.5) -> float:
    return float(stage1_score(labels, probabilities_to_labels(probs, threshold)))

def apply_recipe(p_base, p_aux, *, family: str, alpha: float, margin):
    p_base = np.asarray(p_base, dtype=np.float64)
    p_aux = np.asarray(p_aux, dtype=np.float64)
    blended = (1.0 - alpha) * p_base + alpha * p_aux
    if family == "simple":
        return blended
    if family == "uncertainty_gate":
        if margin is None:
            raise ValueError("uncertainty_gate requires margin")
        gate = np.abs(p_base - 0.5) <= float(margin)
        return np.where(gate, blended, p_base)
    raise ValueError(f"unknown family: {family}")

rows = []
for model_name, wide in oof_tables.items():
    labels = wide["label"].astype(str).to_numpy()
    p_base = wide["p_base"].to_numpy(dtype=np.float64)
    p_aux = wide[f"p_{model_name}"].to_numpy(dtype=np.float64)
    base_score = macro_f1(labels, p_base, THRESHOLD)
    base_fold_scores = []
    for fold in sorted(wide["fold"].unique()):
        mask = wide["fold"].eq(fold).to_numpy()
        base_fold_scores.append(macro_f1(labels[mask], p_base[mask], THRESHOLD))
    base_min_fold = min(base_fold_scores)

    families = [("simple", None)] + [("uncertainty_gate", m) for m in MARGINS]
    for family, margin in families:
        for alpha in ALPHAS:
            p = apply_recipe(p_base, p_aux, family=family, alpha=float(alpha), margin=margin)
            score = macro_f1(labels, p, THRESHOLD)
            fold_scores = []
            for fold in sorted(wide["fold"].unique()):
                mask = wide["fold"].eq(fold).to_numpy()
                fold_scores.append(macro_f1(labels[mask], p[mask], THRESHOLD))
            rows.append({
                "forensic_model": model_name,
                "family": family,
                "alpha": float(alpha),
                "margin": np.nan if margin is None else float(margin),
                "oof_macro_f1_at_0.5": float(score),
                "base_oof_macro_f1_at_0.5": float(base_score),
                "oof_gain": float(score - base_score),
                "min_fold_f1": float(min(fold_scores)),
                "base_min_fold_f1": float(base_min_fold),
                "worst_fold_drop": float(base_min_fold - min(fold_scores)),
            })

search = pd.DataFrame(rows)
search["oof_guard"] = (
    search["oof_macro_f1_at_0.5"].ge(search["base_oof_macro_f1_at_0.5"])
    & search["worst_fold_drop"].le(MAX_WORST_FOLD_DROP + 1e-12)
)
search = search.sort_values(
    ["oof_guard", "oof_macro_f1_at_0.5", "min_fold_f1", "alpha"],
    ascending=[False, False, False, True], kind="mergesort",
).reset_index(drop=True)
search.to_csv(RUN_DIR / "oof_fusion_search.csv", index=False)
display(search.head(30))


## 4. Transfer OOF recipes to A7 and enforce fixed-val retention


In [ ]:
a7 = load_pred(A7_VAL_PATH, "a7")
a7_eval_input = a7.rename(columns={"p_a7": "prob_rerecorded"})
a7_eval = evaluate_predictions(a7_eval_input, threshold=0.5, search_threshold=False)
A7_VAL_SCORE = float(a7_eval.macro_f1_at_default)
print("A7 fixed-val F1 @0.5:", A7_VAL_SCORE)

retention_rows = []
selected = None
selected_val = None

for _, candidate in search.loc[search["oof_guard"]].iterrows():
    model_name = str(candidate["forensic_model"])
    val_path = FORENSIC_ROOT / model_name / "val_predictions.csv"
    if not val_path.is_file():
        continue
    aux = load_pred(val_path, model_name)
    wide = a7.merge(
        aux, on=["video_id", "label", "dataset"], how="inner", validate="one_to_one"
    )
    if len(wide) != len(a7):
        raise RuntimeError(f"{model_name}: fixed-val coverage mismatch")
    family = str(candidate["family"])
    alpha = float(candidate["alpha"])
    margin = None if pd.isna(candidate["margin"]) else float(candidate["margin"])
    p_a7 = wide["p_a7"].to_numpy(dtype=np.float64)
    p_aux = wide[f"p_{model_name}"].to_numpy(dtype=np.float64)
    p = apply_recipe(p_a7, p_aux, family=family, alpha=alpha, margin=margin)

    pred_a7 = np.asarray(probabilities_to_labels(p_a7, 0.5), dtype=object)
    pred_fused = np.asarray(probabilities_to_labels(p, 0.5), dtype=object)
    num_flips = int(np.sum(pred_a7 != pred_fused))
    fused_frame = wide[["video_id", "label", "dataset"]].copy()
    fused_frame["prob_rerecorded"] = p
    fused_frame["prob_original"] = 1.0 - p
    result = evaluate_predictions(fused_frame, threshold=0.5, search_threshold=False)
    drift = np.abs(p - p_a7)
    retention = {
        **candidate.to_dict(),
        "fixed_val_macro_f1_at_0.5": float(result.macro_f1_at_default),
        "a7_fixed_val_macro_f1_at_0.5": A7_VAL_SCORE,
        "num_hard_label_flips_vs_a7": num_flips,
        "mean_abs_probability_drift": float(drift.mean()),
        "max_abs_probability_drift": float(drift.max()),
        "retention_guard": bool(
            result.macro_f1_at_default >= A7_VAL_SCORE - 1e-12 and num_flips == 0
        ),
    }
    retention_rows.append(retention)
    if retention["retention_guard"] and selected is None:
        selected = retention
        selected_val = result.predictions.copy()

retention_table = pd.DataFrame(retention_rows)
retention_table.to_csv(RUN_DIR / "fixed_val_retention.csv", index=False)
display(retention_table.head(30))
if selected is None:
    print("NO FUSION RECIPE SELECTED: no OOF-safe candidate preserved all A7 fixed-val hard labels.")
else:
    print("SELECTED:")
    print(json.dumps(selected, indent=2, default=str))


## 5. Save selected recipe and fixed-val prediction table


In [ ]:
if selected is not None:
    recipe = {
        "experiment": "11_a7_forensic_fusion",
        "git_commit": GIT_COMMIT,
        "anchor": "A7_vjepa2_1_b",
        "forensic_model": selected["forensic_model"],
        "family": selected["family"],
        "alpha": float(selected["alpha"]),
        "margin": None if pd.isna(selected["margin"]) else float(selected["margin"]),
        "threshold": 0.5,
        "selection_domain": "09B/10 grouped OOF",
        "retention_domain": "fixed DLC val",
        "oof_macro_f1_at_0.5": float(selected["oof_macro_f1_at_0.5"]),
        "base_oof_macro_f1_at_0.5": float(selected["base_oof_macro_f1_at_0.5"]),
        "oof_gain": float(selected["oof_gain"]),
        "min_fold_f1": float(selected["min_fold_f1"]),
        "a7_fixed_val_macro_f1_at_0.5": float(selected["a7_fixed_val_macro_f1_at_0.5"]),
        "fixed_val_macro_f1_at_0.5": float(selected["fixed_val_macro_f1_at_0.5"]),
        "num_hard_label_flips_vs_a7": int(selected["num_hard_label_flips_vs_a7"]),
        "mean_abs_probability_drift": float(selected["mean_abs_probability_drift"]),
        "max_abs_probability_drift": float(selected["max_abs_probability_drift"]),
        "a7_val_predictions": str(A7_VAL_PATH),
        "forensic_val_predictions": str(
            FORENSIC_ROOT / str(selected["forensic_model"]) / "val_predictions.csv"
        ),
    }
    (RUN_DIR / "fusion_recipe.json").write_text(
        json.dumps(recipe, indent=2, default=str), encoding="utf-8"
    )
    save_predictions(selected_val, RUN_DIR / "val_predictions.csv")
    print(json.dumps(recipe, indent=2))
    print("saved to:", RUN_DIR)
else:
    diagnostic = {
        "experiment": "11_a7_forensic_fusion",
        "selected": False,
        "reason": "No OOF-nondegrading recipe passed the A7 fixed-val retention gate.",
    }
    (RUN_DIR / "fusion_recipe.json").write_text(
        json.dumps(diagnostic, indent=2), encoding="utf-8"
    )


## After 11

Do not build the submission wrapper until 11 has a selected recipe and its
probability drift is understood. A final multi-branch submission must reproduce
A7 FP32 preprocessing, native-resolution forensic patch extraction, the exact
fusion recipe, and threshold 0.5. Packaging/runtime verification should be a
separate step.
